# Μάθημα 12 - Μείωση Ιστορικού Συνομιλίας με Agent Scratchpad

Αυτό το σημειωματάριο δείχνει πώς να διαχειρίζεστε το πλαίσιο σε μακροχρόνιες συνομιλίες χρησιμοποιώντας το Microsoft Agent Framework. Καθώς οι συνομιλίες μεγαλώνουν, ο αριθμός των tokens αυξάνεται — τελικά υπερβαίνοντας το παράθυρο πλαισίου του μοντέλου. Αντιμετωπίζουμε αυτό με ένα **μοτίβο συνοπτικής παρουσίασης πλαισίου** και ένα **agent scratchpad** για διαρκή μνήμη.

## Τι θα μάθετε:
1. **Γιατί η Διαχείριση Πλαισίου έχει Σημασία**: Κατανόηση ορίων tokens και παραθύρων πλαισίου
2. **Πράκτορες με Ενημέρωση Πλαισίου**: Δημιουργία πρακτόρων που διαχειρίζονται το δικό τους πλαίσιο συνομιλίας
3. **Μοτίβο Συνοπτικής Παρουσίασης Πλαισίου**: Χρήση εργαλείων για συμπύκνωση του ιστορικού συνομιλίας
4. **Agent Scratchpad**: Διαρκής μνήμη που επιβιώνει της μείωσης πλαισίου

## Προαπαιτούμενα:
- Ρύθμιση Azure OpenAI με διαμορφωμένες μεταβλητές περιβάλλοντος
- Κατανόηση βασικών εννοιών πρακτόρων από προηγούμενα μαθήματα


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Γιατί η Διαχείριση του Πλαισίου Είναι Σημαντική

Κάθε LLM έχει ένα πεπερασμένο **παράθυρο πλαισίου** — τον μέγιστο αριθμό token που μπορεί να επεξεργαστεί σε ένα μόνο αίτημα. Καθώς εξελίσσεται μια πολυστροφή συνομιλία:

- **Ο αριθμός των token αυξάνεται γραμμικά** με κάθε μήνυμα χρήστη και απάντηση βοηθού.
- **Τα token του προτροπής κυριαρχούν στο κόστος** γιατί ολόκληρο το ιστορικό ξαναστέλνεται κάθε γύρο.
- Τελικά η συνομιλία **ξεπερνά το παράθυρο πλαισίου** και το μοντέλο είτε περικόπτει είτε παρουσιάζει σφάλμα.

### Στρατηγικές για τη Διαχείριση του Πλαισίου

| Στρατηγική | Πώς Λειτουργεί | Αντίβαρο |
|---|---|---|
| **Περικοπή** | Απορρίπτει τα παλαιότερα μηνύματα | Χάνει νωρίτερο πλαίσιο |
| **Περίληψη** | Συμπυκνώνει παλαιότερα μηνύματα σε μια περίληψη | Χάνονται κάποιες λεπτομέρειες, αλλά διατηρούνται τα βασικά σημεία |
| **Scratchpad / Εξωτερική Μνήμη** | Αποθηκεύει βασικά στοιχεία εκτός συνομιλίας | Απαιτεί κλήσεις εργαλείων, αλλά διατηρείται ανεξαρτήτως περικοπής |

Σε αυτό το σημειωματάριο συνδυάζουμε την **περίληψη** με ένα **εργαλείο scratchpad** ώστε ο πράκτορας να μπορεί να διατηρεί τη συνέχεια ακόμα και όταν το ιστορικό συνομιλίας συμπυκνώνεται.


## Δημιουργία ενός Πράκτορα με Ενημέρωση Συμφραζομένων


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Προσομοίωση μιας Μακράς Συζήτησης

Ας περάσουμε από μια συζήτηση πολλαπλών γύρων για να δούμε πώς συσσωρεύεται το πλαίσιο. Ο πράκτορας πρέπει να διατηρεί βασικές λεπτομέρειες (προτιμήσεις, προϋπολογισμό, ημερομηνίες ταξιδιού) κατά τη διάρκεια των γύρων και να δείχνει συνέχεια.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Παρατηρήστε πώς ο πράκτορας διατηρεί το πλαίσιο από προηγούμενες ανταλλαγές — γνωρίζει για την Ιαπωνία, το σούσι, τους ναούς, τη φωτογραφία, τον προϋπολογισμό των $3000, τα ταξίδια μόνος, και το χρονικό πλαίσιο του Απριλίου. Σε μια σύντομη συνομιλία αυτό λειτουργεί καλά, αλλά καθώς η συνομιλία μεγαλώνει, ολόκληρο το ιστορικό γίνεται δαπανηρό για επαναποστολή.

Ας συνεχίσουμε τη συνομιλία με περισσότερες ανταλλαγές για να δούμε τη συσσώρευση του πλαισίου:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Πρότυπο Περίληψης Πλαισίου

Καθώς η συνομιλία εξελίσσεται, μπορούμε να χρησιμοποιήσουμε ένα **εργαλείο περίληψης** για να συμπτύξουμε το συσσωρευμένο πλαίσιο σε μια συμπαγή μορφή. Ο πράκτορας καλεί αυτό το εργαλείο για να καταγράψει βασικές προτιμήσεις ώστε, ακόμη κι αν παλιότερα μηνύματα απορριφθούν, οι ουσιώδεις πληροφορίες να διατηρούνται.

Αυτό το πρότυπο αποτελεί το δομικό στοιχείο για πιο εξελιγμένη μείωση ιστορικού:
1. Ο πράκτορας εντοπίζει βασικά γεγονότα από τη συνομιλία
2. Καλεί το εργαλείο περίληψης για να τα διατηρήσει
3. Παλαιότερα μηνύματα μπορούν να αφαιρεθούν με ασφάλεια επειδή η περίληψη καταγράφει τα σημαντικά

Παρακάτω ορίζουμε ένα εργαλείο `summarize_preferences` που μπορεί να καλεί ο πράκτορας για να καταγράψει μια συμπαγή περίληψη του τι έχει μάθει.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να διαχειρίζεστε το πλαίσιο σε συνομιλίες πρακτόρων που διαρκούν πολύ χρησιμοποιώντας το Microsoft Agent Framework:

### Βασικές Έννοιες
- **Τα παράθυρα πλαισίου είναι πεπερασμένα** — κάθε διακριτό στοιχείο (token) στο ιστορικό συνομιλίας κοστίζει χρήματα και προσμετράται στο όριο.
- **Τα εργαλεία σύνοψης** επιτρέπουν στον πράκτορα να συμπυκνώνει το συσσωρευμένο πλαίσιο σε συμπαγείς περιλήψεις, μειώνοντας τη χρήση διακριτών στοιχείων ενώ διατηρεί τις ουσιώδεις πληροφορίες.
- **Τα scratchpads του πράκτορα** παρέχουν μόνιμη εξωτερική μνήμη που διατηρείται παρά τις μειώσεις στη συνομιλία.

### Τι Δημιουργήσατε
- Έναν **πράκτορα με επίγνωση πλαισίου** που διατηρεί τη συνέχεια σε συνομιλίες πολλαπλών γύρων
- Ένα **εργαλείο σύνοψης** (`summarize_preferences`) που καταγράφει βασικές λεπτομέρειες χρήστη σε συμπαγή μορφή
- Μια **συνομιλία πολλαπλών γύρων** που δείχνει διατήρηση πλαισίου και διαχείριση αλλαγών

### Εφαρμογές στον Πραγματικό Κόσμο
- **Ρομπότ Εξυπηρέτησης Πελατών**: Θυμούνται προτιμήσεις σε μακροχρόνιες συνεδρίες υποστήριξης
- **Προσωπικοί Βοηθοί**: Παρακολουθούν συνεχιζόμενα έργα χωρίς να απαιτείται επανεξήγηση του πλαισίου
- **Καθηγητές Εκπαίδευσης**: Διατηρούν την πρόοδο του μαθητή σε πολλές αλληλεπιδράσεις

### Επόμενα Βήματα
- Υλοποίηση ενός πλήρους εργαλείου scratchpad με διατήρηση αρχείων
- Προσθήκη αυτόματης περικοπής ιστορικού μετά τη σύνοψη
- Συνδυασμός με βάσεις δεδομένων διανυσμάτων για αναζήτηση σε σημασιολογική μνήμη
- Δημιουργία πρακτόρων που μπορούν να συνεχίσουν συνομιλίες ημέρες αργότερα με πλήρες πλαίσιο


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση Ευθύνης**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ καταβάλλουμε προσπάθειες για την ακρίβεια, παρακαλούμε να λάβετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη γλώσσα του θεωρείται η επίσημη πηγή. Για κρίσιμες πληροφορίες προτείνεται η επαγγελματική μετάφραση από ανθρώπους. Δεν φέρουμε ευθύνη για οποιεσδήποτε παρεξηγήσεις ή λανθασμένες ερμηνείες προκύψουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
